# 🚦 RoadSign Evaluator — Entrenamiento AVANZADO (versión KAGGLE)

Entrena un modelo especializado de señales verticales europeas/españolas. Versión adaptada a **Kaggle Notebooks** (30h/semana de GPU P100 gratis).

---

## ⚙️ ANTES DE EMPEZAR (en Kaggle)

1. Crea cuenta gratis en **kaggle.com** y **verifica tu teléfono** (Settings → Phone Verification). Sin esto la GPU no se activa.
2. Crea un notebook nuevo (**Create → New Notebook**) y sube este archivo (**File → Import Notebook**), o copia las celdas.
3. En el panel derecho: **Session options → Accelerator → GPU P100**.
4. También en el panel derecho activa **Internet → On** (necesario para descargar el dataset).
5. Menú: **Run → Run all**.

## 🔑 API key de Roboflow (gratis)

Crea cuenta en roboflow.com → Settings → API Keys → copia la Private API Key y pégala en el Paso 2.

## 📥 Al terminar

Los archivos `model.onnx` y `labels.json` quedan en la carpeta de salida. En el panel derecho de Kaggle, pestaña **Output / Data**, los verás y podrás descargarlos con el botón de descarga. Súbelos a la carpeta `models/` de tu repositorio.

## Paso 1 — Verificar GPU e instalar herramientas

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if 'P100' in gpu.stdout or 'Tesla' in gpu.stdout or 'GPU' in gpu.stdout:
    print('✅ GPU activa')
    for line in gpu.stdout.split('\n'):
        if 'P100' in line or 'Tesla' in line:
            print('  ', line.strip())
else:
    print('⚠️ NO HAY GPU. Panel derecho → Accelerator → GPU P100')

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'roboflow', 'onnx', 'onnxslim'])
print('\n✅ Herramientas instaladas')

## Paso 2 — Descargar el dataset de señales europeas

**Traffic Signs Detection Europe** (4.381 imágenes, 55 clases). Pega tu API key de Roboflow.

Nota: en Kaggle el panel derecho debe tener **Internet → On**.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TU_API_KEY = "TU_API_KEY"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import os
# En Kaggle trabajamos en /kaggle/working (la única carpeta con escritura persistente)
os.chdir('/kaggle/working')

from roboflow import Roboflow
rf = Roboflow(api_key=TU_API_KEY)
project = rf.workspace("radu-oprea-r4xnm").project("traffic-signs-detection-europe")
dataset = project.version(14).download("yolov8", location='/kaggle/working/dataset_europe')

yaml_path = os.path.join(dataset.location, 'data.yaml')
print(f'\n✅ Dataset en: {dataset.location}')
with open(yaml_path) as f:
    print('\n--- data.yaml ---'); print(f.read())

## Paso 3 — Entrenar (YOLOv8s, preciso)

En GPU P100 tarda aprox. 1-2 horas. Para gastar menos cuota puedes bajar a `yolov8n.pt` y/o reducir epochs.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')   # 'yolov8n.pt' = más rápido y menos cuota

results = model.train(
    data=yaml_path,
    epochs=150, patience=25, imgsz=640, batch=16,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10, translate=0.1, scale=0.5,
    fliplr=0.0, mosaic=1.0, mixup=0.15,
    optimizer='auto', lr0=0.01, cos_lr=True,
    project='/kaggle/working/train', name='europe_signs', exist_ok=True,
)
print('\n✅ Entrenamiento completado')

## Paso 4 — Métricas

In [ ]:
best = YOLO('/kaggle/working/train/europe_signs/weights/best.pt')
metrics = best.val(data=yaml_path)
print(f'📊 mAP50: {metrics.box.map50:.3f} | mAP50-95: {metrics.box.map:.3f}')
print(f'   Precisión: {metrics.box.mp:.3f} | Recall: {metrics.box.mr:.3f}')

## Paso 5 — Exportar a ONNX (queda en /kaggle/working para descargar)

In [ ]:
import shutil, json, os
os.chdir('/kaggle/working')

onnx_path = best.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
shutil.move(onnx_path, '/kaggle/working/model.onnx')

names = best.names
labels = [names[i] for i in range(len(names))]
with open('/kaggle/working/labels.json', 'w', encoding='utf-8') as f:
    json.dump(labels, f, ensure_ascii=False, indent=2)

# Mapeo sugerido de clases → catálogo DGT
lines = ['// Revisa y ajusta signType según tu catálogo', 'const GENERATED_MAP = {']
for i, name in enumerate(labels):
    n = name.lower()
    if 'forbidd' in n or 'prohib' in n or 'no-' in n: cat, st = 'prohibicion', 'R100'
    elif 'warning' in n or 'danger' in n: cat, st = 'peligro', 'P18'
    elif 'mandator' in n or 'oblig' in n: cat, st = 'obligacion', 'M501'
    elif 'inform' in n: cat, st = 'informacion', 'S10'
    elif 'stop' in n: cat, st = 'prioridad', 'R2'
    elif 'yield' in n or 'give-way' in n: cat, st = 'prioridad', 'R1'
    elif 'speed' in n or 'limit' in n: cat, st = 'velocidad', 'R203'
    else: cat, st = 'desconocido', 'UNKNOWN'
    lines.append(f"  {i}: {{ signType:'{st}', category:'{cat}' }},  // {name}")
lines.append('};')
with open('/kaggle/working/classMapper_generado.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

size_mb = os.path.getsize('/kaggle/working/model.onnx')/1024/1024
print(f'✅ Generados en /kaggle/working/:')
print(f'   model.onnx ({size_mb:.1f} MB)')
print(f'   labels.json ({len(labels)} clases)')
print(f'   classMapper_generado.txt')
print('\n📥 DESCARGA: panel derecho → pestaña Output → botón de descarga en cada archivo')
print('   Sube model.onnx y labels.json a la carpeta models/ de tu repo.')